# Step 0: Compare the source DWH extracts (2025 vs 2026) and locate type problems

After switching the step 1 source from `p1_pts_parse_whole_spilt_2025(_)` to `p1_pts_parse_whole_spilt_2026`,
the notebooks fail with type-related errors, for example
`TypeError: '<=' not supported between instances of 'str' and 'float'` in `min()`.
That error means that an `object` column holds **strings and numbers together**.
This happens when a column that used to be numeric (e.g. `RESULT_NUM`) or a date (e.g. `DIAGNOSIS_DATE_2`, `QUERY_DATE`)
arrives as strings in some batch files.

This notebook reads **Parquet metadata only** for the schema comparison, so it is fast. It reads values only for a sample of rows,
except in section 6 (full concatenation, limited to the suspicious columns).

| Section | What it shows |
|---|---|
| 1 | Batch files found in each extract |
| 2 | Columns whose Arrow type changed, differs across batch files, or is missing |
| 3 | Which batch files (d00, d01, ...) carry which type |
| 4 | Value conventions used by step 1 / 2 (`STOP_FLG == '0'`, `ANTI_CANCER_FLG == '＋'`, `'  NEUT#'`, `'M'/'F'`, dates, numerics) |
| 5 | Sampled values, old vs new, and non-numeric strings in numeric columns |
| 6 | Reproduce the step 1 concatenation / Parquet write for the suspicious columns |
| 7 | Check the `step1_all_*.parquet` files that step 2 reads (the source of the `min()` error) |


In [ ]:
import sys, os, platform
sys.path.insert(0, os.path.abspath("../src"))
import pandas as pd, numpy as np, pyarrow as pa
from fnpred import schema_check as sc
pd.set_option("display.max_colwidth", 120); pd.set_option("display.max_columns", 30); pd.set_option("display.width", 250)
print("python", platform.python_version(), sys.executable)
print("pandas", pd.__version__, "| numpy", np.__version__, "| pyarrow", pa.__version__)

In [ ]:
ROOT = "../data/step1_dwh_outputs"
REPORT_DIR = f"{ROOT}/schema_check_2025_vs_2026"
os.makedirs(REPORT_DIR, exist_ok=True)
print("extracts found:", [os.path.basename(p) for p in sc.discover_extracts(ROOT)])

def pick(prefix):
    """step 1 uses '..._2025_' (trailing underscore) for the cancer registry and '..._2025' for the others."""
    cands = [f"{ROOT}/{prefix}", f"{ROOT}/{prefix}_"]
    return [c for c in cands if os.path.isdir(c)]

OLD, NEW = {}, {}
for t in sc.TABLES:
    for tag, prefix, dst in [("old", "p1_pts_parse_whole_spilt_2025", OLD), ("new", "p1_pts_parse_whole_spilt_2026", NEW)]:
        hits = [c for c in pick(prefix) if sc.find_files(c, t)]
        dst[t] = hits[0] if hits else (pick(prefix) or [f"{ROOT}/{prefix}"])[0]
pd.DataFrame({"old": OLD, "new": NEW})

## 1. Batch files per table

In [ ]:
files = pd.DataFrame({t: {"old_files": len(sc.find_files(OLD[t], t)), "new_files": len(sc.find_files(NEW[t], t)),
                           "old_rows": sum(pq_rows for pq_rows in (sc.file_schema(p)["rows"] for p in sc.find_files(OLD[t], t).values())),
                           "new_rows": sum(pq_rows for pq_rows in (sc.file_schema(p)["rows"] for p in sc.find_files(NEW[t], t).values()))}
                       for t in sc.TABLES}).T
files

## 2. Schema differences (columns used by step 1)

In [ ]:
cmp = sc.compare_schemas(OLD, NEW)
issues = sc.issues_overview(cmp, only_used=True)
issues.to_csv(f"{REPORT_DIR}/schema_issues_used_columns.csv", index=False)
for t, m in cmp.items():
    m.to_csv(f"{REPORT_DIR}/schema_{t}.csv", index=False)
issues

In [ ]:
# All columns (including those step 1 does not select), for reference
sc.issues_overview(cmp, only_used=False)

## 3. Which batch files carry which type (for columns with mixed types)

In [ ]:
for _, r in issues[issues["status"].str.contains("mixed|changed")].iterrows():
    print(f"=== {r.table}.{r.column}  ({r.status})")
    print("  old:", sc.batches_by_type(OLD[r.table], r.table, r.column).to_dict("records"))
    print("  new:", sc.batches_by_type(NEW[r.table], r.table, r.column).to_dict("records"))

## 4. Value conventions relied on by step 1 / step 2
`pct_ok_as_read` evaluates the comparison exactly as the code does (e.g. `STOP_FLG == '0'`).
If it drops to ~0% while `pct_ok_as_string` stays high, only the type changed (e.g. `'0'` became `0`), and rows are **silently dropped** rather than raising an error.

In [ ]:
exp = []
for t in sc.EXPECTATIONS:
    o = sc.check_expectations(OLD[t], t).assign(extract="old")
    n = sc.check_expectations(NEW[t], t).assign(extract="new")
    exp.append(pd.concat([o, n]).assign(table=t))
exp = pd.concat(exp).pivot_table(index=["table", "column", "expectation"], columns="extract",
                                 values=["pct_ok_as_string", "pct_ok_as_read"], aggfunc="first")
exp.to_csv(f"{REPORT_DIR}/value_conventions.csv"); exp

## 5. Sampled values, old vs new (suspicious columns), and non-numeric strings

In [ ]:
suspicious = issues[issues["column"] != "<files>"].groupby("table")["column"].apply(list).to_dict()
for t, cols in suspicious.items():
    print(f"=== {t}")
    display(sc.compare_values(OLD[t], NEW[t], t, cols).set_index("column").T)

In [ ]:
NUMERIC = {"MDV_SAMPLE_RSLT": ["RESULT_NUM", "AGE"], "MDV_VITAL": ["TEMPR", "PULSE", "RESP", "BPH", "BPL"],
           "MDV_SOMATOMETRY": ["MS_HEIGHT", "MS_WEIGHT"], "MDV_INJECTION": ["EXEC_DOSE", "AGE"],
           "MDV_DRUG_INSTRUCTION": ["ORDER_DOSE", "AGE"], "CANCER_REGISTRATION": ["AGE"]}
for t, cols in NUMERIC.items():
    for c in cols:
        if c not in suspicious.get(t, []):
            continue
        nv = sc.non_numeric_values(NEW[t], t, c)
        print(f"{t}.{c}: {nv.attrs}")
        display(nv.head(15))

## 6. Reproduce the step 1 concatenation for the suspicious columns (new extract)
Reads **all batch files** of the suspicious columns only. `to_parquet_error` shows the column that makes step 1 fail at `df_all.to_parquet(...)`.

In [ ]:
concat_reports = {}
for t, cols in suspicious.items():
    keep = list(dict.fromkeys(["hs"] + [c for c in cols if c in sc.TABLES[t]]))
    if len(keep) <= 1:
        continue
    concat_reports[t] = sc.try_step1_concat(NEW[t], t, columns=keep)
    print(f"=== {t}"); display(concat_reports[t])
    concat_reports[t].to_csv(f"{REPORT_DIR}/concat_{t}.csv", index=False)

## 7. The `step1_all_*.parquet` files read by step 2
`mixed_types=True` or `pct_numeric_parseable < 100` on `RESULT_NUM`, `TEMPR`, `MS_HEIGHT`, `EXEC_DOSE`, ... explains
`TypeError: '<=' not supported between instances of 'str' and 'float'` in the `min()` / `max()` aggregations of step 2.
Dates stored as strings in `DIAGNOSIS_DATE_2` / `QUERY_DATE` cause the same error in comparisons.

In [ ]:
s2 = sc.check_step2_inputs(ROOT)
s2.to_csv(f"{REPORT_DIR}/step2_inputs.csv", index=False)
s2

## 8. Proposed fix (after identifying the cause)
Cast every batch to one fixed type before `pd.concat` in each step 1 notebook:
```python
from fnpred.schema_check import harmonize
df_list.append(harmonize(df_exam))     # instead of df_list.append(df_exam)
```
`harmonize` casts:
- codes and flags to `string` (removing a trailing `.0` from float-typed codes);
- doses, lab values and vital signs to `float64` (non-numeric strings such as `'<0.1'` become NaN; check section 5 first);
- `QUERY_DATE` / `SOMA_DATE` to `datetime64[ns]`.

Value conventions that changed (section 4, e.g. `STOP_FLG` 0 vs '0', '+' vs '＋') must be handled explicitly in the step 1 / step 2 filters.

## 9. Units and value scale (old vs new)
The Grade 4 / FN thresholds (0.5 / 1.0) assume ANC in 10^3/µL. If the unit or the median changes by ~1,000x, the thresholds must be adapted.

In [ ]:
u_old = sc.lab_unit_summary(OLD["MDV_SAMPLE_RSLT"]).assign(extract="old")
u_new = sc.lab_unit_summary(NEW["MDV_SAMPLE_RSLT"]).assign(extract="new")
units = pd.concat([u_old, u_new]); units.to_csv(f"{REPORT_DIR}/lab_units.csv", index=False); units

In [ ]:
pd.concat({"inj_old": sc.dose_unit_summary(OLD["MDV_INJECTION"]), "inj_new": sc.dose_unit_summary(NEW["MDV_INJECTION"]),
           "oral_old": sc.dose_unit_summary(OLD["MDV_DRUG_INSTRUCTION"], "MDV_DRUG_INSTRUCTION", "ORDER_UNIT"),
           "oral_new": sc.dose_unit_summary(NEW["MDV_DRUG_INSTRUCTION"], "MDV_DRUG_INSTRUCTION", "ORDER_UNIT")}, axis=1)